In [ ]:
# Find the project root from either the workspace or notebooks folder.
from pathlib import Path
from datetime import datetime, timezone
import json
import sys
PROJECT_ROOT = next((folder for folder in (Path.cwd(), *Path.cwd().parents)
                     if (folder / "src/task_b").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open the NLP_PROJECT_task_b folder in VS Code before running this notebook")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Point to independently reviewed paired sentence and atomic annotations.
import pandas as pd
from task_b.custom import load_reviewed_pairs
from task_b.schema import LABELS
from task_b.verification import lexical_cues, timed_predictions, evaluate, save_run

REVIEWED = PROJECT_ROOT / "data/annotations/task_b_reviewed_pairs.jsonl"
OUTPUT = PROJECT_ROOT / "outputs/task_b/atomic_vs_sentence" / datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%SZ")
assert REVIEWED.is_file(), "Create an independently reviewed custom set first"


In [ ]:
# Validate that each sentence and its atoms share the same answer and source.
sentences, atoms = load_reviewed_pairs(REVIEWED)
print("Reviewed sentences:", len(sentences), "reviewed atoms:", len(atoms))


In [ ]:
# Apply one fixed verifier to both levels; keep their scores separate.
results = {}
for level, rows in (("sentence", sentences), ("atomic", atoms)):
    predictions, latency = timed_predictions([row[0] for row in rows], lexical_cues)
    score = evaluate([row[1] for row in rows], predictions, LABELS, latency)
    score["evaluation_scope"] = "independently reviewed custom set: " + level
    results[level] = score
    save_run(OUTPUT / level, {"method": "lexical_cues", "level": level, "threshold": 0.5, "reviewed_source": str(REVIEWED)}, [{"claim_id": row[0].claim_id, "answer_id": row[2], "document_id": row[3], "gold": row[1], "prediction": prediction, "latency_seconds": elapsed} for row, prediction, elapsed in zip(rows, predictions, latency)], score)


In [ ]:
# Inspect three-class scores without pooling sentence and atomic examples.
display(pd.DataFrame([{"level": level, "n": score["n"], "accuracy": score["accuracy"], "macro_f1": score["macro_f1"], "contradicted_recall": score["contradicted_recall"], "incorrect_supported": score["incorrect_supported"], "mean_latency_ms": score["latency_mean_ms"]} for level, score in results.items()]))
for level, score in results.items():
    print(level)
    display(pd.DataFrame(score["per_class"]).T)
    display(pd.DataFrame(score["confusion_matrix"], index=score["labels"], columns=score["labels"]))
